# nb_silver_to_gold
clean the messy `sales_raw_bronze.csv` into a Delta table.

In [ ]:
run_date = "2026-06-13"

In [ ]:
from pyspark.sql import functions as F

# If a date doesn't match a format, return NULL instead of an error
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

raw = (spark.read.option("header", True).option("inferSchema", True)
       .csv("Files/bronze/sales_raw_bronze.csv"))

raw_count = raw.count()
print("Raw rows:", raw_count)
display(raw.limit(10))

In [ ]:
# Cell 3: clean
clean = (raw
    # fix types and spaces
    .withColumn("unit_price",   F.trim(F.col("unit_price")).cast("double"))
    .withColumn("quantity",     F.col("quantity").cast("int"))
    .withColumn("net_amount",   F.col("net_amount").cast("double"))
    .withColumn("discount_pct", F.col("discount_pct").cast("int"))

    # ONLINE / online / " Online " -> Online
    .withColumn("channel", F.initcap(F.trim(F.col("channel"))))

    # two date formats -> one DATE column
    .withColumn("order_date", F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "dd-MM-yyyy")))

    # remove duplicate orders
    .dropDuplicates(["order_id"])

    # rebuild a missing net_amount, tand then a missing quantity   (F.coalesce(A, B) means “use A, but if A is blank, use B”)
    .withColumn("net_amount", F.coalesce(
        F.col("net_amount"),
        F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100), 2)))
    .withColumn("quantity", F.coalesce(
        F.col("quantity"),
        F.round(F.col("net_amount") / (F.col("unit_price") * (1 - F.col("discount_pct") / 100))).cast("int")))

    # Keep only rows where both quantity and net_amount have values
    .filter(F.col("quantity").isNotNull() & F.col("net_amount").isNotNull())

    # revenue band
    .withColumn("revenue_band",
        F.when(F.col("net_amount") < 500, "Low <500")
         .when(F.col("net_amount") <= 2000, "Mid 500-2000")
         .otherwise("High >2000"))

    .withColumn("_loaded_at", F.current_timestamp())
)

In [ ]:
# save as a Delta table and check the result
clean.write.mode("overwrite").format("delta").saveAsTable("silver_sales")

silver_count = spark.table("silver_sales").count()
print("Raw rows:   ", raw_count)
print("Silver rows:", silver_count)
print("Removed:    ", raw_count - silver_count)